# Annual Character Count

Aggregate cleaned text volume by year and write the results to `output/annual_character_counts.csv`.

In [ ]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "data_pipeline" else Path.cwd()
TEXT_ROOT = ROOT / "output" / "plain_text_cleaned"
OUTPUT_CSV = ROOT / "output" / "annual_character_counts.csv"


def iter_text_files(root: Path):
    for path in sorted(root.rglob("*.txt")):
        if ".ipynb_checkpoints" in path.parts:
            continue
        if path.name.endswith("-checkpoint.txt"):
            continue
        yield path


rows = []
for path in iter_text_files(TEXT_ROOT):
    year = int(path.parts[-3])
    text = path.read_text(encoding="utf-8", errors="ignore")
    rows.append(
        {
            "year": year,
            "doc_id": path.stem,
            "character_count": len(text),
        }
    )

df = pd.DataFrame(rows)
annual = (
    df.groupby("year", as_index=False)
    .agg(
        document_count=("doc_id", "count"),
        total_character_count=("character_count", "sum"),
        average_character_count=("character_count", "mean"),
    )
    .sort_values("year")
)
annual["average_character_count"] = annual["average_character_count"].round(2)

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
annual.to_csv(OUTPUT_CSV, index=False)

annual